# Análisis Exploratorio de Datos (EDA): Dataset PubHealth

## Objetivos del análisis

En este notebook se analiza el conjunto de datos PubHealth para la detección de desinformación en el ámbito de la salud. Los objetivos son:

- Comprender la estructura y calidad de los datos.
- Detectar desbalanceo de clases.
- Analizar patrones textuales (longitud, vocabulario) que diferencien noticias falsas de verdaderas.

## Configuración del entorno

### Definir importaciones necesarias

In [ ]:
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# Asegurar que el módulo src.utils.load_data esté en el path
sys.path.append('../utils')
from load_data import load_dataset

### Configuración visual para las gráficas

In [ ]:
sns.set_theme(style="whitegrid")

## Carga y limpieza de datos

Se realiza una comprobación de valores nulos y duplicados para garantizar la calidad del entrenamiento. Las filas sin texto o etiquetas se descartan.

### Visualización inicial de los datos

In [ ]:
df = load_dataset()

print(f"Dimensiones: {df.shape}")
print("-" * 30)
print(df.info())
print("-" * 30)
display(df.head())

### Detección y tratamiento de valores nulos

In [ ]:
# Comprobar nulos
null_values = df.isnull().sum()
if null_values.sum() == 0:
    print("No hay valores nulos en el dataset")
else:
    print("Valores nulos por columna:")
    print(null_values[null_values > 0])

# Eliminar filas con nulos en las columnas 'claim' o 'label'
rows = len(df)
df = df.dropna(subset=['claim', 'label'])
print(f"Filas tras limpiar nulos: {len(df)} (Se eliminaron {rows - len(df)} filas)")

### Detección y tratamiento de valores duplicados

In [ ]:
# Comprobar duplicados en la columna 'claim'
duplicate_count = df.duplicated(subset=['claim']).sum()
if duplicate_count == 0:
	print("No hay noticias duplicadas en la columna 'claim'")
else:
	print(f"Número de noticias duplicadas: {duplicate_count}")

# Eliminar duplicados
df_clean = df.drop_duplicates(subset=['claim'])
print(f"Filas tras eliminar duplicados: {len(df_clean)} (Se eliminaron {len(df) - len(df_clean)} filas)")

## Balanceo de clases

Analizamos la distribución de las clases para detectar posibles desbalanceos que afecten al entrenamiento del modelo.

### Mapeo de etiquetas a categorías

In [ ]:
# Asegurar que la columna sea string antes de mapear para evitar errores de tipo
df_clean['label'] = df_clean['label'].astype(str)

etiquetas_map = {
    '0': 'Verdadero',
    '1': 'Falso',
    '2': 'Mezcla',
    '3': 'Sin comprobar'
}

# Crear una columna nueva legible para las gráficas
df_clean['label_text'] = df_clean['label'].map(etiquetas_map)

print(f"Clases originales: {df_clean['label'].unique()}")
print(f"Clases transformadas: {df_clean['label_text'].unique()}")

### Analizar si el dataset está balanceado.

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='label_text', data=df_clean, order=df_clean['label_text'].value_counts().index, palette='viridis', hue='label_text', legend=False)

plt.title('Distribución de clases (veracidad)', fontsize=14)
plt.xlabel('Etiqueta')
plt.ylabel('Número de noticias')

# Mostrar el número encima de cada barra
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.show()

## Análisis estructural del texto

Estudiamos si la longitud de las noticias influye en su veracidad.

### Distribución de la longitud de los textos por clase

In [ ]:
# Calcular la longitud en palabras (aproximada)
df_clean['word_count'] = df_clean['claim'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(df_clean['word_count'], bins=50, kde=True, color='purple')

plt.title('Distribución de longitud de las noticias (palabras)', fontsize=14)
plt.xlabel('Número de Palabras')
plt.xlim(0, 100)
plt.show()

print(f"Longitud media: {df_clean['word_count'].mean():.2f} palabras")
print(f"Longitud máxima: {df_clean['word_count'].max()} palabras")

### Relación entre longitud del titular y veracidad

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='label_text', y='word_count', data=df_clean, palette='Set2', hue='label_text', legend=False)
plt.title('Distribución de longitud del texto por veracidad', fontsize=14)
plt.xlabel('Veracidad')
plt.ylabel('Número de palabras')
plt.ylim(0, 100)
plt.show()

## Análisis semántico y de contenido

### Distribución de temas

In [ ]:
# Aplanar la lista de temas
all_subjects = []
for subjects in df_clean['subjects'].dropna():
    if isinstance(subjects, str):
        # Limpiar y separar por comas
        subjects_list = [t.strip().lower() for t in subjects.split(',')]
        all_subjects.extend(subjects_list)

# Contar los temas más frecuentes
subjects_count = Counter(all_subjects)
df_subjects = pd.DataFrame(subjects_count.most_common(10), columns=['Tema', 'Frecuencia'])

# Graficar
plt.figure(figsize=(10, 6))
sns.barplot(x='Frecuencia', y='Tema', data=df_subjects, palette='magma', hue='Tema', legend=False)
plt.title('Top 10 temáticas en las noticias', fontsize=14)
plt.show()

### Nubes de palabras por clase

In [ ]:
# Crear subplots
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# Nube de noticias falsas
subset_fake = df_clean[df_clean['label_text'] == 'Falso']['claim'].dropna().astype(str)
text_fake = " ".join(subset_fake)
wc_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(text_fake)
axs[0].imshow(wc_fake, interpolation='bilinear')
axs[0].set_title('Vocabulario en noticias falsas', fontsize=16)
axs[0].axis('off')

# Nube de noticias verdaderas
subset_true = df_clean[df_clean['label_text'] == 'Verdadero']['claim'].dropna().astype(str)
text_true = " ".join(subset_true)
wc_true = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(text_true)
axs[1].imshow(wc_true, interpolation='bilinear')
axs[1].set_title('Vocabulario en noticias verdaderas', fontsize=16)
axs[1].axis('off')

plt.show()

## Conclusiones

Tras el análisis exhaustivo de las variables y la estructura del dataset **PubHealth**, se extraen las siguientes conclusiones determinantes para la fase de modelado:

### 1. Integridad y calidad del dato
El dataset presenta una **calidad excepcional**, evidenciada por la ausencia total de valores nulos (`NaN`) y registros duplicados en las afirmaciones (*claims*).
* **Implicación:** No será necesario aplicar pipelines complejos de imputación de datos o limpieza agresiva durante la fase de preprocesamiento, lo que reduce el riesgo de introducir ruido artificial en el entrenamiento.

### 2. Dimensionamiento de tokens para BERT
El análisis de longitud revela que estamos ante textos breves y concisos:
* **Longitud Media:** 13.58 palabras.
* **Longitud Máxima:** 85 palabras.

**Decisión de ingeniería:**
Dado que los modelos tipo BERT suelen tener un límite de 512 tokens, y nuestra noticia más larga tiene solo 85 palabras (aprox. 100-110 tokens), **podemos reducir la ventana de contexto (`MAX_LENGTH`) a 128 tokens** sin riesgo de truncar información relevante.
* Esto optimizará significativamente el uso de memoria VRAM y acelerará el tiempo de entrenamiento sin perder precisión.

### 3. Viabilidad del proyecto
La brevedad de los textos sugiere que el modelo deberá aprender a detectar desinformación basándose en **patrones semánticos fuertes** y palabras clave (vocabulario alarmista o técnico) en espacios cortos, un escenario ideal para la arquitectura de Atención de los Transformers.